<a href="https://colab.research.google.com/github/linhtutkyawdev/my_aggri_cner_v2/blob/master/my_agri_cner_v2_mt5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Burmese Agriculture CNER: mT5 Fine-Tuning Pipeline (Fold 0 Try-Out)
This notebook is an enhanced, highly-professional fine-tuning pipeline for **mT5 (Multilingual T5)** to perform Burmese Agricultural Concept Named Entity Recognition (CNER). It is adapted from the legacy Colab file and structured identically to the `my_agri_cner_v2.ipynb` pipeline, adopting its clean step-by-step layout, persistent directory management, and robust reporting.

### 🎯 Fold 0 Fast Experimentation
Instead of running a heavy 5-Fold Cross-Validation (which can take days on seq2seq models), this notebook targets **Fold 0** of the standard CoNLL dataset splits (`bioes_word` and `bioes_syllable`). This allows rapid try-out, model validation, and easy comparison with NCRF++ baselines!

### 💾 Colab Disconnect Protection & Real-time Persistence
To protect against Google Colab timeouts, inactivity disconnects, or runtime restarts:
1. **Real-time Persistence**: The model checkpoints (`models_mt5/`) and prediction files (`output_mt5/`) are symlinked and written directly to your **Google Drive** in real-time.
2. **Auto-Skip Completed Experiments**: If a setup's training is already completed and metrics are generated, the execution loop automatically skips it and immediately loads the existing evaluation report. This saves tons of training time!


### Step 1: Connect to Google Drive


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Google Colab environment not detected. Skipping Google Drive mount.")


### Step 2: Configure Environment and Paths


In [ ]:
import os
import sys
import shutil

# Detect Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Project directory in Google Drive
    %env PJ_DIR=/content/drive/My Drive/MyAgriNER_copy
    # Configure PyTorch memory management to prevent fragmentation and Out-Of-Memory errors
    %env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
    print("Colab PJ_DIR and PyTorch memory allocator configured.")

print(f"Running in Colab: {IN_COLAB}")


### Step 3: Clone Repository & Install Dependencies


In [ ]:
# Install the necessary libraries
dependencies = ['transformers', 'datasets', 'sentencepiece', 'protobuf', 'seqeval', 'evaluate', 'accelerate']

print("Installing dependencies...")
!pip install -q {" ".join(dependencies)}


### Step 4: Persistent Google Drive Symlinking & Dataset Extraction


In [ ]:
import os
import shutil

# 1. Define persistent directories in Google Drive / Local
if IN_COLAB:
    drive_root = '/content/drive/My Drive/MyAgriNER'
    drive_data = os.path.join(drive_root, 'data')
    drive_models = os.path.join(drive_root, 'models_mt5')
    drive_output = os.path.join(drive_root, 'output_mt5')

    # Create directories on Google Drive if they don't exist
    os.makedirs(drive_data, exist_ok=True)
    os.makedirs(drive_models, exist_ok=True)
    os.makedirs(drive_output, exist_ok=True)

    # 2. Extract data.zip directly to Google Drive (only if folders are missing to save time!)
    already_extracted = os.path.exists(os.path.join(drive_data, 'bioes_word')) if os.path.exists(drive_data) else False

    if not already_extracted:
        zip_paths = [
            "/content/data.zip",
            "/content/drive/My Drive/MyAgriNER/data.zip",
            "/content/drive/My Drive/data.zip"
        ]
        found_zip = None
        for p in zip_paths:
            if os.path.exists(p):
                found_zip = p
                break

        if found_zip:
            print(f"Found data.zip at: {found_zip}")
            print("Extracting data.zip directly to Google Drive... This may take a minute.")
            !unzip -q -o "{found_zip}" -d "/content/drive/My Drive/MyAgriNER/"
            print("Extraction complete!")
        else:
            print("Warning: data.zip not found! Please upload data.zip to Google Drive (MyAgriNER/) or Colab (/content/).")
    else:
        print("Data directory already exists and contains extracted files on Google Drive. Skipping extraction.")

    # 3. Symlink directories to Google Drive
    for folder, drive_path in [('data', drive_data), ('models_mt5', drive_models), ('output_mt5', drive_output)]:
        local_path = os.path.abspath(folder)
        if os.path.exists(local_path):
            if os.path.islink(local_path):
                os.unlink(local_path)
            else:
                shutil.rmtree(local_path)
        os.symlink(drive_path, local_path)
        print(f'Symlinked local {local_path} -> persistent Google Drive: {drive_path}')
else:
    # Local paths configuration
    os.makedirs('data', exist_ok=True)
    os.makedirs('models_mt5', exist_ok=True)
    os.makedirs('output_mt5', exist_ok=True)
    print("Running locally. Output directories set to: models_mt5/, output_mt5/")


### Step 5: Robust K-Fold Dataset Splitter (Rotating Blocks)


In [ ]:
import os
from pathlib import Path

def load_conll_sentences(file_path):
    """Parses a CoNLL file and returns a list of raw sentence blocks."""
    sentences = []
    current_sentence = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            current_sentence.append(line)
            if not line.strip():
                if current_sentence:
                    sentences.append("".join(current_sentence))
                    current_sentence = []
        if current_sentence:
            content = "".join(current_sentence)
            if not content.endswith('\n'):
                content += '\n'
            if not content.endswith('\n\n'):
                content += '\n'
            sentences.append(content)
    return sentences

def split_sentences_by_ratio(sentences, ratio, folds):
    """Splits sentences into folds using rotating block selection."""
    B = sum(ratio)
    N = len(sentences)
    X, Y, Z = ratio

    folds_data = []
    for i in range(folds):
        S_i = int(i * B / folds)

        # Rotating block indices modulo B
        train_blocks = [(S_i + b) % B for b in range(X)]
        dev_blocks = [(S_i + X + b) % B for b in range(Y)]
        test_blocks = [(S_i + X + Y + b) % B for b in range(Z)]

        def get_sentences_for_blocks(blocks):
            selected = []
            for b in sorted(blocks):
                start_idx = int(b * N / B)
                end_idx = int((b + 1) * N / B)
                selected.extend(sentences[start_idx:end_idx])
            return selected

        train_sents = get_sentences_for_blocks(train_blocks)
        dev_sents = get_sentences_for_blocks(dev_blocks)
        test_sents = get_sentences_for_blocks(test_blocks)

        folds_data.append((train_sents, dev_sents, test_sents))
    return folds_data

def process_file(file_path, output_dir, ratio, folds):
    print(f"Processing CoNLL file: '{file_path}'")
    sentences = load_conll_sentences(file_path)
    print(f"  Loaded {len(sentences)} sentences.")

    folds_data = split_sentences_by_ratio(sentences, ratio, folds)
    base_name = Path(file_path).stem

    setup_dir = Path(output_dir) / base_name
    setup_dir.mkdir(parents=True, exist_ok=True)

    for i, (train, dev, test) in enumerate(folds_data):
        fold_dir = setup_dir / f"fold_{i}"
        fold_dir.mkdir(parents=True, exist_ok=True)

        with open(fold_dir / "train.conll", 'w', encoding='utf-8') as f:
            f.writelines(train)
        with open(fold_dir / "dev.conll", 'w', encoding='utf-8') as f:
            f.writelines(dev)
        with open(fold_dir / "test.conll", 'w', encoding='utf-8') as f:
            f.writelines(test)

        print(f"    Fold {i} -> Train: {len(train)} sents, Dev: {len(dev)} sents, Test: {len(test)} sents")

data_dir = 'data'

# Check if splits are already present
has_splits = False
if os.path.exists(data_dir):
    subdirs = [os.path.join(data_dir, d) for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    for subdir in subdirs:
        if 'fold_0' in os.listdir(subdir):
            has_splits = True
            break

if not has_splits:
    conll_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir)
                   if f.endswith('.conll') and not f.startswith(('train.', 'dev.', 'test.'))]
    for file in sorted(conll_files):
        process_file(file, data_dir, [8, 1, 1], 5)
    print('\nAll datasets split into 5 folds successfully!')
else:
    print('Dataset splits already exist. Skipping splitting phase!')


### Step 6: Dataset Loader & mT5 Serialization


In [ ]:
import os

def load_conll_data(file_path):
    """Parses standard CoNLL file into sentences with 'tokens' and 'labels'."""
    sentences = []
    current_tokens = []
    current_labels = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_tokens:
                    sentences.append({"tokens": current_tokens, "labels": current_labels})
                    current_tokens = []
                    current_labels = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    current_tokens.append(parts[0])
                    current_labels.append(parts[1])
        if current_tokens:
            sentences.append({"tokens": current_tokens, "labels": current_labels})
    return sentences

def clean_label(label):
    """Strips B-, I-, E-, S- prefixes to match mT5 brackets training format."""
    if "-" in label:
        return label.split("-", 1)[1]
    return label

def serialize_mt5(tokens: list, labels: list) -> dict:
    """Converts lists of tokens and labels into mT5 input_text and target_text."""
    input_text = "cner: " + " ".join(tokens)
    
    target_parts = []
    for token, label in zip(tokens, labels):
        cleaned = clean_label(label)
        if cleaned == "O":
            target_parts.append(token)
        else:
            target_parts.append(f"{token}[{cleaned}]")
            
    target_text = " ".join(target_parts)
    
    return {
        "tokens": tokens,
        "labels": labels,
        "input_text": input_text,
        "target_text": target_text
    }

def prepare_mt5_fold_datasets(setup_name, fold_idx, data_dir='data'):
    """Loads a specific fold, cleans tags, and serializes it to mT5 format."""
    fold_path = os.path.join(data_dir, setup_name, f"fold_{fold_idx}")
    
    train_file = os.path.join(fold_path, "train.conll")
    dev_file = os.path.join(fold_path, "dev.conll")
    test_file = os.path.join(fold_path, "test.conll")
    
    train_sents = load_conll_data(train_file)
    dev_sents = load_conll_data(dev_file)
    test_sents = load_conll_data(test_file)
    
    train_serialized = [serialize_mt5(s["tokens"], s["labels"]) for s in train_sents]
    dev_serialized = [serialize_mt5(s["tokens"], s["labels"]) for s in dev_sents]
    test_serialized = [serialize_mt5(s["tokens"], s["labels"]) for s in test_sents]
    
    print(f"[{setup_name} Fold {fold_idx}] Loaded: Train={len(train_serialized)}, Dev={len(dev_serialized)}, Test={len(test_serialized)}")
    return train_serialized, dev_serialized, test_serialized


### Step 7: Sequence-to-Sequence CNER Evaluator


In [ ]:
import re
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

def parse_mt5_output(text: str) -> list:
    """Parses a generated mT5 string back into a list of dictionaries with 'text' and 'label'."""
    entities = []
    parts = text.strip().split()
    for part in parts:
        match = re.match(r"^(.+?)\[([A-Z_]+)\]$", part)
        if match:
            token = match.group(1)
            label = match.group(2)
            entities.append({"text": token, "label": label})
        else:
            entities.append({"text": part, "label": "O"})
    return entities

def evaluate_predictions(references: list, predictions: list) -> dict:
    """Calculates entity-level precision, recall, and F1 metrics using seqeval."""
    true_sequences = []
    pred_sequences = []
    invalid_count = 0
    exact_match_count = 0
    
    for ref_str, pred_str in zip(references, predictions):
        ref_parsed = parse_mt5_output(ref_str)
        pred_parsed = parse_mt5_output(pred_str)
        
        if ref_str.strip() == pred_str.strip():
            exact_match_count += 1
            
        true_lbls = []
        for item in ref_parsed:
            lbl = item["label"]
            if lbl != "O":
                true_lbls.append(f"B-{lbl}")
            else:
                true_lbls.append("O")
                
        pred_lbls = []
        if len(pred_parsed) != len(ref_parsed):
            invalid_count += 1
            for i in range(len(ref_parsed)):
                if i < len(pred_parsed):
                    lbl = pred_parsed[i]["label"]
                    if lbl != "O":
                        pred_lbls.append(f"B-{lbl}")
                    else:
                        pred_lbls.append("O")
                else:
                    pred_lbls.append("O")
        else:
            for item in pred_parsed:
                lbl = item["label"]
                if lbl != "O":
                    pred_lbls.append(f"B-{lbl}")
                else:
                    pred_lbls.append("O")
                    
        true_sequences.append(true_lbls)
        pred_sequences.append(pred_lbls)
        
    has_pred = any(any(l != "O" for l in seq) for seq in pred_sequences)
    has_true = any(any(l != "O" for l in seq) for seq in true_sequences)
    
    p = precision_score(true_sequences, pred_sequences) if has_pred else 0.0
    r = recall_score(true_sequences, pred_sequences) if has_true else 0.0
    f1 = f1_score(true_sequences, pred_sequences) if (p + r) > 0 else 0.0
    
    return {
        "precision": p * 100,
        "recall": r * 100,
        "f1": f1 * 100,
        "exact_match_count": exact_match_count,
        "invalid_generated_count": invalid_count
    }


### Step 8: Execution Master Loop - Fine-Tune mT5 and Evaluate on Fold 0 (With Auto-Skip Checkpoints)


In [ ]:
import os, json, torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, TrainerCallback
from datasets import Dataset

# --- CONFIGURATION ---
model_name = "google/mt5-base"   # Defaulting to high-accuracy mt5-base!
epochs = 5
train_batch_size = 2     # Set to 2 (paired with gradient_accumulation_steps=8 for maximum VRAM safety)
eval_batch_size = 2      # Set to 2 for maximum VRAM safety
learning_rate = 1e-4 if "base" in model_name.lower() else 3e-4
weight_decay = 0.01
warmup_ratio = 0.1
max_input_length = 256   # 256 to prevent truncation
max_target_length = 256  # 256 to prevent truncation
seed = 42
setups = ["bioes_word", "bioes_syllable"]
fold_idx = 0
results = {}

# Set seeds
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

cuda_available = torch.cuda.is_available()
use_fp16 = False
use_bf16 = cuda_available and torch.cuda.is_bf16_supported()
print(f"Precision Config: FP16={use_fp16}, BF16={use_bf16}")

class ProductionLoggingCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\\n--- Epoch {state.epoch:.1f} / {args.num_train_epochs} Finished ---")
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print(f"  Step {state.global_step} | Training Loss: {logs['loss']:.4f}")
        if logs and "eval_loss" in logs:
            print(f"  --- Validation Metrics (Step {state.global_step}) ---")
            print(f"    Eval Loss: {logs['eval_loss']:.4f}")
            for key in ["precision", "recall", "f1", "exact_match_count", "invalid_generated_count"]:
                if f"eval_{key}" in logs:
                    print(f"    {key.replace('_', ' ').title()}: {logs[f'eval_{key}']:.2f}")
            print("  --------------------------------------------\\n")

for setup in setups:
    experiment_name = f"{setup}_fold_{fold_idx}_mt5"
    output_dir = f"models_mt5/{experiment_name}"
    metrics_path = f"output_mt5/{experiment_name}_metrics.json"
    predictions_path = f"output_mt5/{experiment_name}_predictions.json"
    
    print("="*70 + f"\\n💎 EXPERIMENT: {setup} (Fold {fold_idx}) using {model_name}\\n" + "="*70)
    
    if os.path.exists(metrics_path):
        print(f"Metrics report for {experiment_name} already exists. Skipping training and restoring results...")
        with open(metrics_path, "r", encoding="utf-8") as f:
            metrics_report = json.load(f)
        results[setup] = metrics_report["final_test_metrics"]
        print(f"  [Restored] Test F1-Score: {results[setup]['test_f1']:.2f}%")
        continue
        
    # Forceful Garbage Collection & VRAM cache flushing
    import gc
    print("🧹 Cleaning up VRAM from any crashed or previous failed runs...")
    for var in ["model", "trainer", "tokenizer"]:
        if var in globals():
            del globals()[var]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    train_records, dev_records, test_records = prepare_mt5_fold_datasets(setup, fold_idx)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.config.use_cache = False
    
    preprocess_fn = lambda x: {"labels": tokenizer(text_target=x["target_text"], max_length=max_target_length, truncation=True)["input_ids"], **tokenizer(x["input_text"], max_length=max_input_length, truncation=True)}
    train_tokenized = Dataset.from_list(train_records).map(preprocess_fn, batched=True, remove_columns=["input_text", "target_text", "tokens", "labels"])
    val_tokenized = Dataset.from_list(dev_records).map(preprocess_fn, batched=True, remove_columns=["input_text", "target_text", "tokens", "labels"])
    test_tokenized = Dataset.from_list(test_records).map(preprocess_fn, batched=True, remove_columns=["input_text", "target_text", "tokens", "labels"])
    
    steps_per_epoch = max(1, len(train_records) // train_batch_size)
    warmup_steps = int(steps_per_epoch * epochs * warmup_ratio)
    
    # Auto-resume: Check for existing checkpoints to continue seamlessly
    checkpoints_dir = os.path.join(output_dir, "checkpoints")
    resume_checkpoint = None
    if os.path.exists(checkpoints_dir):
        checkpoints = [os.path.join(checkpoints_dir, d) for d in os.listdir(checkpoints_dir) if d.startswith("checkpoint-")]
        if checkpoints:
            checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
            resume_checkpoint = checkpoints[-1]
            print(f"  🔄 [Auto-Resume] Found existing checkpoints. Continuing training from step: {os.path.basename(resume_checkpoint)}")
    
    is_base = "base" in model_name.lower()
    grad_accum = 8 if (is_base and cuda_available) else 1
    use_checkpointing = True if (is_base and cuda_available) else False
    
    training_args = Seq2SeqTrainingArguments(
        output_dir=checkpoints_dir,
        eval_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        weight_decay=weight_decay,
        save_total_limit=1,
        num_train_epochs=epochs,
        predict_with_generate=True,
        logging_steps=max(1, steps_per_epoch // 2),
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        fp16=use_fp16,
        bf16=use_bf16,
        use_cpu=not cuda_available,
        warmup_steps=warmup_steps,
        gradient_accumulation_steps=grad_accum,
        gradient_checkpointing=use_checkpointing,
        report_to="none",
        dataloader_num_workers=2 if cuda_available else 0,
        disable_tqdm=False,
        optim="adafactor", # Switched to memory-efficient Adafactor optimizer
    )
    
    def compute_metrics(eval_preds):
        preds, labels_ids = eval_preds
        if isinstance(preds, tuple): preds = preds[0]
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        labels_ids = np.where(labels_ids != -100, labels_ids, tokenizer.pad_token_id)
        decoded_preds = [p.replace("<pad>", "").replace("</s>", "").replace("<unk>", "").strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=False)]
        decoded_labels = [l.replace("<pad>", "").replace("</s>", "").replace("<unk>", "").strip() for l in tokenizer.batch_decode(labels_ids, skip_special_tokens=False)]
        return evaluate_predictions(decoded_labels, decoded_preds)
        
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
        compute_metrics=compute_metrics,
        callbacks=[ProductionLoggingCallback()],
    )
    
    print("Starting production fine-tuning...")
    trainer.train(resume_from_checkpoint=resume_checkpoint)

    best_model_dir = os.path.join(output_dir, "best_model")
    print(f"Saving best model to {best_model_dir}...")
    trainer.save_model(best_model_dir)
    tokenizer.save_pretrained(best_model_dir)
    
    print("Running evaluation on Unseen Test Dataset...")
    test_metrics = trainer.evaluate(eval_dataset=test_tokenized, metric_key_prefix="test")
    
    print("Generating test predictions report...")
    test_preds_output = trainer.predict(test_dataset=test_tokenized, max_length=max_target_length)
    test_preds_ids = test_preds_output.predictions
    if isinstance(test_preds_ids, tuple): test_preds_ids = test_preds_ids[0]
    test_preds_ids = np.where(test_preds_ids != -100, test_preds_ids, tokenizer.pad_token_id)
    
    decoded_test_preds = [p.replace("<pad>", "").replace("</s>", "").replace("<unk>", "").strip() for p in tokenizer.batch_decode(test_preds_ids, skip_special_tokens=False)]
    decoded_test_labels = [r["target_text"] for r in test_records]
    
    predictions_report = [{"input": r["input_text"], "gold": ref, "pred": pred} for r, ref, pred in zip(test_records, decoded_test_labels, decoded_test_preds)]
    os.makedirs(os.path.dirname(predictions_path), exist_ok=True)
    with open(predictions_path, "w", encoding="utf-8") as f:
        json.dump(predictions_report, f, indent=4, ensure_ascii=False)
        
    final_metrics = {
        "test_loss": test_metrics.get("test_test_loss"),
        "test_precision": test_metrics.get("test_precision"),
        "test_recall": test_metrics.get("test_recall"),
        "test_f1": test_metrics.get("test_f1"),
        "test_exact_match_count": test_metrics.get("test_exact_match_count"),
        "test_invalid_generated_count": test_metrics.get("test_invalid_generated_count"),
    }
    
    metrics_report = {
        "config": {"base_model": model_name, "epochs": epochs, "learning_rate": learning_rate, "train_batch_size": train_batch_size, "eval_batch_size": eval_batch_size},
        "final_test_metrics": final_metrics
    }
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics_report, f, indent=4)
        
    results[setup] = final_metrics
    print(f"💎 Setup {setup} Complete! Test F1: {final_metrics['test_f1']:.2f}%\\n")

print("\\n" + "="*95 + "\\n 🏆 mT5 FOLD 0 TRY-OUT PERFORMANCE REPORT 🏆\\n" + "="*95)
print(f"{'Setup':<20} | {'Entity Precision':^18} | {'Entity Recall':^18} | {'Entity F1-Score':^18} | {'Exact Matches':^18} |")
print("-"*95)
for setup, metrics in results.items():
    print(f"{setup:<20} | {metrics['test_precision']:16.2f}% | {metrics['test_recall']:16.2f}% | {metrics['test_f1']:16.2f}% | {metrics['test_exact_match_count']:18,} |")
print("="*95)


### Step 9: Backup All Experiments to Google Drive


In [ ]:
from datetime import datetime

if IN_COLAB:
    drive_export_path = '/content/drive/My Drive/MyAgriNER/'
    mt5_backup_path = os.path.join(drive_export_path, 'mT5_backup')
    os.makedirs(mt5_backup_path, exist_ok=True)

    src_dir = 'models_mt5'
    dst_dir = os.path.join(mt5_backup_path, datetime.now().strftime('%Y%m%d_%H%M%S'))

    print(f'Starting backup of {src_dir} to {dst_dir}...')

    if os.path.exists(src_dir):
        shutil.copytree(src_dir, dst_dir)
        print(f'\nBackup complete! Saved in {dst_dir}.')
    else:
        print(f'Error: Source directory {src_dir} does not exist.')
else:
    print("Running locally. Skipping Google Drive backup step.")


### Step 10: Interactive Burmese Agricultural CNER Predictor
This cell allows you to run live, real-time predictions on any custom Burmese text! It loads your best fine-tuned mT5 model directly from Google Drive or local workspace storage, runs generation with beam search, and formats the entities inside brackets.


In [ ]:
# Interactive Burmese Agricultural CNER Predictor
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Select the setup to use for predictions
setup_name = "bioes_word"  # Or "bioes_syllable"
best_model_path = f"models_mt5/{setup_name}_fold_0_mt5/best_model"

if not os.path.exists(best_model_path):
    print(f"❌ Error: Fine-tuned model not found at '{best_model_path}'. Please run Step 8 first!")
else:
    print(f"🎯 Loading best fine-tuned model from: {best_model_path}")
    tokenizer = AutoTokenizer.from_pretrained(best_model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(best_model_path)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    
    def predict_cner(text):
        input_text = "cner: " + text.strip()
        inputs = tokenizer(input_text, return_tensors="pt", max_length=256, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=256, num_beams=4, early_stopping=True)
        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        return prediction

    # Run on sample custom texts
    sample_text = "စပါး စိုက်ပျိုး ရာတွင် ဂျစ်ဆန် နှင့် ယူရီးယား ကို ၂ကြိမ် ခွဲ၍သုံးပါ"
    predicted_brackets = predict_cner(sample_text)
    print("\n📝 Custom Test Prediction:")
    print(f"  Input:      {sample_text}")
    print(f"  Prediction: {predicted_brackets}")


### Step 11: Visual Test Set Inspector (Side-by-Side Comparison)
This inspector randomly selects 5 sentences from your unseen test fold, executes the best fine-tuned model on them, and displays a beautiful side-by-side comparison between the raw text, ground-truth targets, and model predictions to let you evaluate the actual predictive quality visually!


In [ ]:
# Visual Test Set Inspector (Side-by-Side Comparison)
import random

if not os.path.exists(best_model_path):
    print(f"❌ Error: Fine-tuned model not found at '{best_model_path}'. Please run Step 8 first!")
else:
    # Load fold_0 test records
    _, _, test_records = prepare_mt5_fold_datasets(setup_name, fold_idx=0)
    
    # Select 5 random test samples
    random.seed(42)
    samples = random.sample(test_records, min(5, len(test_records)))
    
    print(f"\n🧐 VISUAL INSPECTION: 5 RANDOM TEST SAMPLES ({setup_name})")
    print("=" * 95)
    for idx, sample in enumerate(samples, 1):
        raw_input = sample["input_text"].replace("cner: ", "").strip()
        gold = sample["target_text"].strip()
        pred = predict_cner(raw_input)
        
        print(f"\n[Sample #{idx}]")
        print(f"  Input text:   {raw_input}")
        print(f"  Ground Truth: {gold}")
        print(f"  Model Pred:   {pred}")
        print("-" * 95)
